# TMP: Degenerate LV investigation

**Environment:** `clamp-analyses`

Identifies LVs that inflate trait coverage counts in the Hallmarks random-subsampling GLS results.
Two types of degenerate LVs were found:

1. **Zero p-value LVs**: p-value underflows to 0.0 for many traits, directly producing FDR=0 associations.
2. **Anomalously broad LVs**: non-zero but extremely small p-values for hundreds of traits (implausible biology).

Both types inflate the global BH correction, boosting apparent trait coverage beyond what the model warrants.

In [1]:
library(here)
library(dplyr)
library(readr)

base_output_dir <- here("output/01_model_building/04_archs4/06_bp_coverage_rshall")

GetModelSpec <- function(path) {
  model_dir <- basename(dirname(path))
  pieces <- regmatches(model_dir, regexec("^hall_coverage_rs([0-9]+)_seed_([0-9]+)$", model_dir))[[1]]
  list(
    model        = model_dir,
    coverage_pct = as.integer(pieces[2]),
    seed         = as.integer(pieces[3])
  )
}

summary_paths <- list.files(
  base_output_dir,
  pattern = "gls-summary-phenomexcan\\.tsv\\.gz$",
  recursive = TRUE,
  full.names = TRUE
)
summary_paths <- summary_paths[grepl("/hall_coverage_rs[0-9]+_seed_[0-9]+/gls-summary-phenomexcan\\.tsv\\.gz$", summary_paths)]
summary_paths <- summary_paths[!grepl("/hall_coverage_rs100_seed_[0-9]+/", summary_paths)]
summary_paths <- sort(summary_paths)

stopifnot(length(summary_paths) == 18)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Type 1: LVs with zero p-values (floating-point underflow)

In [2]:
zero_pval_summary <- lapply(summary_paths, function(path) {
  spec <- GetModelSpec(path)
  df <- readr::read_tsv(path, show_col_types = FALSE, progress = FALSE)

  df %>%
    dplyr::filter(pvalue == 0) %>%
    dplyr::group_by(lv) %>%
    dplyr::summarise(n_zero_pvalue = dplyr::n(), .groups = "drop") %>%
    dplyr::mutate(coverage_pct = spec$coverage_pct, seed = spec$seed, model = spec$model)
}) %>%
  dplyr::bind_rows() %>%
  dplyr::arrange(coverage_pct, seed, dplyr::desc(n_zero_pvalue))

print(zero_pval_summary)

# A tibble: 5 × 5
  lv    n_zero_pvalue coverage_pct  seed model                    
  <chr>         <int>        <int> <int> <chr>                    
1 LV834           645           10     1 hall_coverage_rs10_seed_1
2 LV509           644           10     2 hall_coverage_rs10_seed_2
3 LV593           643           10     3 hall_coverage_rs10_seed_3
4 LV519           643           50     1 hall_coverage_rs50_seed_1
5 LV836           641           50     2 hall_coverage_rs50_seed_2


## Type 2: LVs with anomalously many significant traits (FDR < 0.05)

Threshold: LVs with > 200 significant traits are flagged.
For reference, the max in clean models is ~132 (50% seed_3) and ~118 (25% seed_1).

In [3]:
BROAD_LV_THRESHOLD <- 200L

broad_lv_summary <- lapply(summary_paths, function(path) {
  spec <- GetModelSpec(path)
  df <- readr::read_tsv(path, show_col_types = FALSE, progress = FALSE)

  df %>%
    dplyr::filter(fdr < 0.05) %>%
    dplyr::group_by(lv) %>%
    dplyr::summarise(n_sig_traits = dplyr::n_distinct(phenotype), .groups = "drop") %>%
    dplyr::filter(n_sig_traits > BROAD_LV_THRESHOLD) %>%
    dplyr::mutate(coverage_pct = spec$coverage_pct, seed = spec$seed, model = spec$model)
}) %>%
  dplyr::bind_rows() %>%
  dplyr::arrange(coverage_pct, seed, dplyr::desc(n_sig_traits))

print(broad_lv_summary)

# A tibble: 9 × 5
  lv    n_sig_traits coverage_pct  seed model                    
  <chr>        <int>        <int> <int> <chr>                    
1 LV4            398            1     3 hall_coverage_rs1_seed_3 
2 LV834         1889           10     1 hall_coverage_rs10_seed_1
3 LV448          472           10     1 hall_coverage_rs10_seed_1
4 LV509         1889           10     2 hall_coverage_rs10_seed_2
5 LV593         1889           10     3 hall_coverage_rs10_seed_3
6 LV519         1889           50     1 hall_coverage_rs50_seed_1
7 LV836         1888           50     2 hall_coverage_rs50_seed_2
8 LV186          229           75     1 hall_coverage_rs75_seed_1
9 LV855          367           75     3 hall_coverage_rs75_seed_3


## Summary of degenerate LVs to exclude

Known degenerate LVs identified (used as TMP filters in `04_trait_coverage.ipynb`):

| Coverage | Seed | LV | Type | n_affected_traits |
|----------|------|----|------|-------------------|
| 10% | all | LV834 | zero p-value | ~645 |
| 10% | 2 | LV509 | anomalously broad | 1889 |
| 10% | 3 | LV593 | anomalously broad | 1889 |
| 50% | 1, 2 | LV519 | zero p-value | ~642 |
| 50% | 2 | LV836 | anomalously broad | 1888 |

## Head of traits for the ~1889-trait degenerate LVs

In [4]:
broad_lvs <- list(
  list(path = summary_paths[grepl("rs10_seed_2", summary_paths)], lv = "LV509",  label = "10% seed_2: LV509"),
  list(path = summary_paths[grepl("rs10_seed_3", summary_paths)], lv = "LV593",  label = "10% seed_3: LV593"),
  list(path = summary_paths[grepl("rs50_seed_2", summary_paths)], lv = "LV836",  label = "50% seed_2: LV836")
)

for (entry in broad_lvs) {
  cat("\n====", entry$label, "====\n")
  df <- readr::read_tsv(entry$path, show_col_types = FALSE, progress = FALSE)
  top <- df %>%
    dplyr::filter(lv == entry$lv, fdr < 0.05) %>%
    dplyr::select(phenotype, phenotype_desc, pvalue, fdr) %>%
    dplyr::arrange(pvalue) %>%
    head(20)
  print(top)
}


==== 10% seed_2: LV509 ====
# A tibble: 20 × 4
   phenotype  phenotype_desc               pvalue   fdr
   <chr>      <chr>                         <dbl> <dbl>
 1 100002_raw Energy                            0     0
 2 100004_raw Fat                               0     0
 3 100005_raw Carbohydrate                      0     0
 4 100006_raw Saturated fat                     0     0
 5 100008_raw Total sugars                      0     0
 6 100009_raw Englyst dietary fibre             0     0
 7 100013_raw Vitamin B12                       0     0
 8 100015_raw Vitamin C                         0     0
 9 100019_raw Carotene                          0     0
10 100021_raw Vitamin D                         0     0
11 100025_raw Vitamin E                         0     0
12 100250     Instant coffee intake             0     0
13 100260     Added milk to instant coffee      0     0
14 100630     Rose wine intake                  0     0
15 100730     Spirits intake                    0     0
